# recs_013 — Heuristic ranker candidates (learning notebook)

Read-only guide to **Candidate A–G**: the D1-style formulas we will compare on frozen `two_tower_v1` pools.

**You are here because:** before tuning seven variants, it helps to know *what kind* of rerank each one is — not just the formula.

**Implementation notebook:** [`recs_013_ranker_d1_heuristic.ipynb`](recs_013_ranker_d1_heuristic.ipynb) (Candidates A–G).

**Broader ranker context:** [`recs_013_ranker_approaches_learn.ipynb`](recs_013_ranker_approaches_learn.ipynb) (D1–D6), [`docs/ranker_exploration_plan.md`](../../docs/ranker_exploration_plan.md).

## 1. What we are doing (one sentence)

Retrieval already gave us **top-100** games per user. A **heuristic ranker** re-scores those 100 items with a hand-written formula, sorts again, and shows **top-10**.

Each **candidate** is a different formula. Each has **hyperparameters** (e.g. `alpha`). We **grid-search on train pools**, pick the best config per candidate, then **face off on val** (no val tuning).

## 2. Four "flavors" (how to think about candidates)

Candidates are not random tweaks. They fall into four families:

| Flavor | Idea | Candidates |
|--------|------|------------|
| **A. Score blend** | Combine retrieval score + popularity on the same scale, then sort | A, C, D |
| **B. Rank fusion** | Ignore raw magnitudes; fuse *positions* in two orderings | B |
| **C. Constrained rerank** | Only rerank part of the pool, or only when a rule fires | E, F |
| **D. Sanity baseline** | Deliberately extreme recipe to interpret results | G |

**Why flavors matter:** if A and C both win, popularity *transform* (log vs raw) mattered. If B beats A, score scale was hurting you. If E wins, you should not rerank the whole pool.

## 3. Shared inputs (every candidate uses these)

| Signal | Where it comes from |
|--------|---------------------|
| `retrieval_score` | Frozen pool from `recs_job_export_retrieval_pools.py` / val jsonl |
| `popularity` | Global train positive-review count per game (`pop_row`) |
| Labels | Held-out positives in the pool row (for NDCG only — not a feature) |

**Within-pool normalization:** most score-blend candidates min-max each signal *inside the 100 items* so one outlier score does not dominate.

**Not allowed in this phase:** tuning on val, re-running two-tower, or using label info in the score (except oracle, for headroom only).

## 4. Candidate cards

### Candidate A — Linear min-max blend *(implemented)*

**Flavor:** score blend

**Formula:** `score = α·norm(retr) + (1−α)·norm(pop)`

**Params:** `α ∈ [0, 1]`

**Intuition:** Simple knob between "trust retrieval" (α→1) and "trust popularity" (α→0). Your current best α≈0.2 says a little retrieval + a lot of pop helps within the pool.

**Method name:** `two_tower_v1_heuristic_pop_blend`

---

### Candidate B — Reciprocal rank fusion (RRF)

**Flavor:** rank fusion

**Formula:** `score = w_r/(k + rank_retr) + w_p/(k + rank_pop)`  
(`rank` = 1 for best item in that ordering; higher pop → better → lower rank number)

**Params:** weights `w_r`, `w_p`; constant `k` (often 60)

**Intuition:** Does not care if retrieval scores are 0.99 vs 0.98 — only *order*. Robust when score scales are weird. Common in search when merging ranked lists.

**Method name:** `two_tower_v1_heuristic_rrf`

---

### Candidate C — Log-pop linear blend

**Flavor:** score blend (popularity transform)

**Formula:** `score = α·norm(retr) + (1−α)·norm(log(1+pop))`

**Params:** `α`

**Intuition:** Same as A, but compresses mega-hit games. Stops one extremely popular title from swamping mid-tier retrieval hits.

**Method name:** `two_tower_v1_heuristic_logpop_blend`

---

### Candidate D — Multiplicative (geometric) blend

**Flavor:** score blend (different geometry)

**Formula:** `score = norm(retr)^α · norm(pop)^(1−α)`

**Params:** `α`

**Intuition:** *Both* signals must be decent; one near-zero drags the product down. Additive A can rescue an item with high pop + low retr; multiplicative cannot.

**Method name:** `two_tower_v1_heuristic_geom_blend`

---

### Candidate E — Top-M retrieval, then blend

**Flavor:** constrained rerank

**Formula:** Apply Candidate A **only** to the top-M items by retrieval; keep retrieval order for the rest.

**Params:** `α`, `M` (e.g. 20, 50, 100)

**Intuition:** Protect strong retrieval hits in positions 1–M; let popularity shuffle the middle. Good if full-pool rerank hurts obvious matches.

**Method name:** `two_tower_v1_heuristic_pop_topm`

---

### Candidate F — Gated pop boost

**Flavor:** constrained rerank

**Formula:** `score = norm(retr) + β·norm(pop)` **only if** `norm(retr) ≥ τ`; else `norm(retr)`

**Params:** boost `β`, threshold `τ`

**Intuition:** Popularity helps items retrieval already liked a bit; it cannot override clear retrieval losers.

**Method name:** `two_tower_v1_heuristic_pop_gated`

---

### Candidate G — Popularity-only within pool

**Flavor:** sanity baseline

**Formula:** sort pool by `pop` only (same as A with α=0)

**Params:** none

**Intuition:** How much lift is *pure* popularity rerank vs any retrieval-aware recipe? Compare to global `popularity_train` baseline story.

**Method name:** `two_tower_v1_heuristic_pop_only`

## 5. Selection protocol (no winging it)

For each candidate C:
1. Grid-search hyperparams on **train** pools → `best_params[C]`
2. Primary metric: mean NDCG@10, Slice A

Val face-off (once): baseline, each candidate @ its best params, oracle.

**Winner:** best val Slice A NDCG among candidates (oracle is headroom, not a competitor).

## 6. Suggested build order

1. **A** — baseline (done)
2. **B, C** — most likely to beat A; different flavor
3. **E, F** — if A/B/C plateau
4. **D, G** — geometry check + sanity row

Refactor pattern in the implementation notebook: one `*_pool_scores(...)` per candidate + shared `mean_ndcg` / val leaderboard.

## 7. Further reading (external)

**Learning to rank (context for why we rerank at all)**
- [Learning to rank — Wikipedia](https://en.wikipedia.org/wiki/Learning_to_rank) — pointwise / pairwise / listwise vocabulary; D1 is "hand pointwise."

**Reciprocal rank fusion (Candidate B)**
- [Mathematical Intuition Behind Reciprocal Rank Fusion](https://medium.com/@devalshah1619/mathematical-intuition-behind-reciprocal-rank-fusion-rrf-explained-in-2-mins-002df0cc5e2a)   .
- [Elasticsearch RRF docs](https://www.elastic.co/guide/en/elasticsearch/reference/current/rrf.html) — practical explanation of `k` and merging ranked lists.

**Popularity bias in recommenders (why C, G matter)**
- [A Survey on Popularity Bias in Recommender Systems](https://arxiv.org/abs/2308.01118)

- [Relieving popularity bias in recommender systems via user group-level augmentation
](https://www.sciencedirect.com/science/article/abs/pii/S156849462401384X)


**Two-stage retrieve → rank (our architecture)**
- [Huang et al. — Embedding-based Retrieval in Facebook Search (EBR)](https://research.facebook.com/publications/embedding-based-retrieval-in-facebook-search/) — industry pattern: cheap retrieval, then rerank.

**In-repo**
- [`docs/recommendation_evaluation_overview.md`](../../docs/recommendation_evaluation_overview.md) — retrieval vs ranking metrics
- [`docs/retrieval_metrics_guide.md`](../../docs/retrieval_metrics_guide.md) — NDCG, Hit@K on Slice A/B

## 8. Toy example — same pool, three flavors

Run the next cell. Six fake pool items: retrieval likes **A** first; positives are **B** and **F**; popularity favors **D/F**.

Watch how **A (blend)**, **B (RRF)**, and **G (pop-only)** reorder top-3.

In [1]:
import numpy as np
import pandas as pd

POSITIVE = {1, 5}  # games B, F

items = pd.DataFrame(
    {
        "game": list("ABCDEF"),
        "retrieval_score": [0.95, 0.55, 0.50, 0.45, 0.42, 0.40],
        "popularity": [0.10, 0.35, 0.20, 0.90, 0.30, 0.85],
    }
)


def minmax(x: np.ndarray) -> np.ndarray:
    lo, hi = float(x.min()), float(x.max())
    return np.zeros_like(x) if hi - lo <= 1e-12 else (x - lo) / (hi - lo)


def show_scores(name: str, scores: np.ndarray) -> None:
    order = np.argsort(-scores)
    top3_idx = order[:3]
    top3 = [items.loc[i, "game"] for i in top3_idx]
    hits = sum(1 for i in top3_idx if i in POSITIVE)
    print(f"{name:22} top-3={top3}  hits@3={hits}")


retr = items["retrieval_score"].to_numpy()
pop = items["popularity"].to_numpy()
alpha = 0.6

show_scores("Retrieval only", retr)
show_scores("A blend α=0.6", alpha * minmax(retr) + (1 - alpha) * minmax(pop))

rank_retr = np.argsort(np.argsort(-retr)) + 1
rank_pop = np.argsort(np.argsort(-pop)) + 1
k = 60
show_scores("B RRF k=60", 1.0 / (k + rank_retr) + 1.0 / (k + rank_pop))

show_scores("G pop only", pop)

print("\nPositives (oracle targets): B, F")

Retrieval only         top-3=['A', 'B', 'C']  hits@3=1
A blend α=0.6          top-3=['A', 'D', 'F']  hits@3=1
B RRF k=60             top-3=['D', 'B', 'A']  hits@3=1
G pop only             top-3=['D', 'F', 'B']  hits@3=2

Positives (oracle targets): B, F


**Takeaway:** retrieval puts **A** first but labels are **B/F**. Blending pulls **E/D** up (popularity). RRF can behave differently because it uses ranks, not score gaps. This is why we compare flavors — not just crank `alpha` on one formula.